In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
import datetime
from keras.datasets import fashion_mnist
import wandb

In [2]:
%load_ext autoreload
%autoreload 2
from Model import NeuralNetwork

In [3]:
def normalize(x):
    return x.reshape(len(x), -1).astype('float64') / (np.max(x) - np.min(x))

In [4]:
def load_and_prepare_data(dataset="fashion_mnist"):
    # Load the Fashion MNIST dataset
    (x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
    
    # Using train_test_split to separate validation data (10% of training data)
    x_train, x_val, y_train, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=69)
    
    # Normalize image pixel values
    x_train = normalize(x_train)
    x_val   = normalize(x_val)
    x_test  = normalize(x_test)
    
    # Determine the number of classes from the unique labels
    classes = np.unique(y_train)
    num_classes = len(classes)
    
    # One-hot encode labels based on the discovered number of classes
    y_train = np.eye(num_classes)[y_train]
    y_val   = np.eye(num_classes)[y_val]
    y_test  = np.eye(num_classes)[y_test]
    
    return x_train, y_train, x_val, y_val, x_test, y_test

In [5]:
def train_and_evaluate(config=None):
    with wandb.init(config=config):
        cfg = wandb.config
        # Create a dynamic name based on hyperparameters
        sweep_name = f"hl_{cfg.num_layers}_hs_{cfg.hidden_size}_bs_{cfg.batch_size}_ac_{cfg.activation}_opt_{cfg.optimizer}_lr_{cfg.learning_rate}"
    
        # Assign the dynamically generated name
        wandb.run.name = sweep_name
        wandb.run.save()
        # Load and prepare the dataset
        x_train, y_train, x_val, y_val, x_test, y_test = load_and_prepare_data()
        
        # Model with configuration parameters
        model = NeuralNetwork(
            input_size = x_train.shape[1],
            num_classes = y_train.shape[1],
            num_hidden = cfg.num_layers,
            hidden_units = cfg.hidden_size,
            init_method = cfg.weight_init,
            activation = cfg.activation,
            loss_fn = cfg.loss,
            epochs = cfg.epochs,
            batch_size = cfg.batch_size,
            optimizer = cfg.optimizer,
            lr = cfg.learning_rate,
            weight_decay = cfg.weight_decay,
            momentum = cfg.momentum if hasattr(cfg, 'momentum') else 0.9,
            beta = cfg.beta if hasattr(cfg, 'beta') else 0.9,
            beta1 = cfg.beta1 if hasattr(cfg, 'beta1') else 0.9,
            beta2 = cfg.beta2 if hasattr(cfg, 'beta2') else 0.999,
            epsilon = cfg.epsilon if hasattr(cfg, 'epsilon') else 1e-6,
            iswandb = True
        )
        
        # Train the model using the training and validation data
        model.fit(x_train, y_train, x_val, y_val)
        
        # Evaluate on validation set
        val_preds = model.predict(x_val.T)
        val_loss  = model.compute_loss(val_preds, y_val)
        val_acc   = model.accuracy(val_preds, y_val)
        
        # Evaluate on test set
        test_preds = model.predict(x_test.T)
        test_loss  = model.compute_loss(test_preds, y_test)
        test_acc   = model.accuracy(test_preds, y_test)
        
        # Log evaluation metrics to wandb with a timestamp
        wandb.log({
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "test_loss": test_loss,
            "test_accuracy": test_acc,
            "created": datetime.datetime.now().isoformat()
        })

In [6]:
sweep_config = {
    'method': 'bayes',
    'name': 'cross_entropy_vs_mean_squared_error',
    'metric': {'name': 'validation_accuracy', 'goal': 'maximize'},
    'parameters': {
        'epochs': {'values': [5, 10]},
        'num_layers': {'values': [3, 4, 5]},
        'hidden_size': {'values': [32, 64, 128]},
        'weight_decay': {'values': [0, 0.0005, 0.5]},
        'learning_rate': {'values': [0.001, 0.0001]},
        'optimizer': {'values': ['sgd', 'momentum', 'nag', 'rmsprop', 'adam', 'nadam']},
        'batch_size': {'values': [16, 32, 64]},
        'weight_init': {'values': ['Random', 'Xavier']},
        'activation': {'values': ['Sigmoid', 'Tanh', 'ReLU']},
        'loss': {'values': ['mean_squared_error']}
    }
}

In [7]:
def run_experiment():
    sweep_id = wandb.sweep(sweep_config, project="fashion-mnist-classification")
    wandb.agent(sweep_id, function=train_and_evaluate, count=30)
    wandb.finish()

In [8]:
if __name__ == "__main__":
    run_experiment()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Create sweep with ID: jnozoefz
Sweep URL: https://wandb.ai/mrsagarbiswas-iit-madras/fashion-mnist-classification/sweeps/jnozoefz


wandb: Agent Starting Run: bousxvjf with config:
wandb: 	activation: Tanh
wandb: 	batch_size: 64
wandb: 	epochs: 5
wandb: 	hidden_size: 32
wandb: 	learning_rate: 0.0001
wandb: 	loss: mean_squared_error
wandb: 	num_layers: 4
wandb: 	optimizer: rmsprop
wandb: 	weight_decay: 0
wandb: 	weight_init: Random
wandb: Currently logged in as: mrsagarbiswas (mrsagarbiswas-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Calling wandb.run.save without any arguments is deprecated.Changes to attributes are automatically persisted.


Epoch 1: train_loss = 1.28, valid_loss = 1.29, train_accuracy = 0.19, val_accuracy = 0.18
Epoch 2: train_loss = 1.15, valid_loss = 1.16, train_accuracy = 0.20, val_accuracy = 0.19
Epoch 3: train_loss = 0.97, valid_loss = 0.98, train_accuracy = 0.23, val_accuracy = 0.22
Epoch 4: train_loss = 0.87, valid_loss = 0.87, train_accuracy = 0.30, val_accuracy = 0.30
Epoch 5: train_loss = 0.81, valid_loss = 0.81, train_accuracy = 0.36, val_accuracy = 0.35


...

Epoch 1: train_loss = 0.69, valid_loss = 0.68, train_accuracy = 0.42, val_accuracy = 0.42
Epoch 2: train_loss = 0.72, valid_loss = 0.71, train_accuracy = 0.41, val_accuracy = 0.41
Epoch 3: train_loss = 0.72, valid_loss = 0.72, train_accuracy = 0.40, val_accuracy = 0.40
Epoch 4: train_loss = 0.73, valid_loss = 0.72, train_accuracy = 0.41, val_accuracy = 0.41
Epoch 5: train_loss = 0.71, valid_loss = 0.71, train_accuracy = 0.44, val_accuracy = 0.44


epoch,▁▃▅▆█
test_accuracy,▁
test_loss,▁
train_accuracy,▅▃▁▃█
train_loss,▁▆██▅
val_accuracy,▅▃▁▂██
val_loss,▁▆██▅▅
created,2025-03-15T20:37:15....
epoch,4
test_accuracy,0.429
test_loss,0.71096


wandb: Agent Starting Run: knylwhzz with config:
wandb: 	activation: Tanh
wandb: 	batch_size: 16
wandb: 	epochs: 5
wandb: 	hidden_size: 128
wandb: 	learning_rate: 0.0001
wandb: 	loss: mean_squared_error
wandb: 	num_layers: 4
wandb: 	optimizer: nadam
wandb: 	weight_decay: 0.5
wandb: 	weight_init: Random


Epoch 1: train_loss = 1.53, valid_loss = 1.53, train_accuracy = 0.13, val_accuracy = 0.13
Epoch 2: train_loss = 1.31, valid_loss = 1.31, train_accuracy = 0.18, val_accuracy = 0.19
Epoch 3: train_loss = 0.79, valid_loss = 0.79, train_accuracy = 0.41, val_accuracy = 0.41
Epoch 4: train_loss = 0.55, valid_loss = 0.56, train_accuracy = 0.59, val_accuracy = 0.59
Epoch 5: train_loss = 0.41, valid_loss = 0.41, train_accuracy = 0.71, val_accuracy = 0.72


epoch,▁▃▅▆█
test_accuracy,▁
test_loss,▁
train_accuracy,▁▂▄▇█
train_loss,█▇▃▂▁
val_accuracy,▁▂▄▆██
val_loss,█▇▃▂▁▁
created,2025-03-15T20:38:42....
epoch,4
test_accuracy,0.7017
test_loss,0.42656
